# 02 ASR Baseline

Common Voice Yue baseline notebook. This version supports a real Whisper API zero-shot baseline through the VectorEngine-compatible OpenAI API, then saves prediction CSVs for error analysis.


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

def running_in_colab():
    return 'google.colab' in sys.modules or Path('/content').exists()



def find_project_root():
    cwd = Path.cwd().resolve()
    env_root = os.getenv('PROJECT_ROOT')
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path('/content/Cantonese-Speech-AI'),
        Path('/content/drive/MyDrive/Cantonese-Speech-AI'),
        Path('/content/drive/My Drive/Cantonese-Speech-AI'),
        Path(r'D:/GitHub/Cantonese-Speech-AI'),
        Path(r'D:\GitHub\Cantonese-Speech-AI'),
        Path('/mnt/d/GitHub/Cantonese-Speech-AI'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'cantonese_speech_ai').exists():
            return candidate.resolve()
    checked = '\n'.join(str(path) for path in candidates[:12])
    raise FileNotFoundError(
        'Cannot find the Cantonese-Speech-AI project folder in this runtime.\n\n'
        'Colab fix:\n'
        '1. Upload or copy the whole Cantonese-Speech-AI folder to Google Drive.\n'
        '2. Expected path: /content/drive/MyDrive/Cantonese-Speech-AI\n'
        '3. The folder must contain src/cantonese_speech_ai and Mozilla-HK-Speech-datasets.\n'
        '4. Then restart runtime and run this cell again.\n\n'
        'Alternative: set os.environ[\'PROJECT_ROOT\'] to your folder path before this cell.\n\n'
        f'Current cwd: {cwd}\nChecked:\n{checked}'
    )

ROOT = find_project_root()
os.environ['PROJECT_ROOT'] = str(ROOT)
load_dotenv(ROOT / '.env')

default_dataset = ROOT / 'Mozilla-HK-Speech-datasets' / 'cv-corpus-26.0-2026-06-12' / 'yue'
os.environ.setdefault('COMMON_VOICE_YUE_ROOT', str(default_dataset))
os.environ.setdefault('VECTORENGINE_BASE_URL', 'https://api.vectorengine.cn/v1')
os.environ.setdefault('WHISPER_MODEL', 'whisper-1')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from cantonese_speech_ai.common_voice import CommonVoicePaths, prepare_split
from cantonese_speech_ai.metrics import char_error_rate, word_error_rate
from cantonese_speech_ai.whisper_api import WhisperApiConfig, build_prediction_row, transcribe_audio

paths = CommonVoicePaths.from_env()
if not paths.split_path('train').exists():
    raise FileNotFoundError(f'Dataset train.tsv not found: {paths.split_path("train")}')
ROOT, paths


(WindowsPath('D:/GitHub/Cantonese-Speech-AI'),
 CommonVoicePaths(root=WindowsPath('D:/GitHub/Cantonese-Speech-AI/Mozilla-HK-Speech-datasets/cv-corpus-26.0-2026-06-12/yue')))

## Small baseline subset

In [2]:
train = pd.DataFrame(prepare_split('train', paths)).head(500)
dev = pd.DataFrame(prepare_split('dev', paths)).head(100)
test = pd.DataFrame(prepare_split('test', paths))

baseline_columns = ['audio_path', 'sentence', 'normalized_sentence', 'duration_sec', 'accents']
train[baseline_columns].head()

,audio_path,sentence,normalized_sentence,duration_sec,accents
0,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,大陸有專俾同志用嘅交友程式？,大陸有專俾同志用嘅交友程式,4.608,广东音
1,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,仲有舊年年尾話唔應該再猛咁撩鼻，不如做抗體檢測好過，個訪問冇耐就喺大陸全網禁咗,仲有舊年年尾話唔應該再猛咁撩鼻 不如做抗體檢測好過 個訪問冇耐就喺大陸全網禁咗,10.080,广州音
2,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,邊有噉大隻蛤乸隨街跳,邊有噉大隻蛤乸隨街跳,3.528,广州音
3,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,但係我唔特登講就成日會俾人估錯隔離,但係我唔特登講就成日會俾人估錯隔離,5.148,广州音
4,D:\GitHub\Cantonese-Speech-AI\Mozilla-HK-Speec...,除非有人喺第二區截車入去,除非有人喺第二區截車入去,3.636,广州音


## Whisper API baseline

Start with a small API smoke test before scaling up. This is a zero-shot baseline, not model fine-tuning. Keep `api_rows` small at first to control cost and debug safely.


In [3]:
model_plan = {
    'dataset': 'Mozilla Common Voice Yue',
    'train_rows': len(train),
    'dev_rows': len(dev),
    'primary_metric': 'CER',
    'baseline_method': 'Whisper API zero-shot transcription',
    'api_base_url': os.environ.get('VECTORENGINE_BASE_URL'),
    'api_model': os.environ.get('WHISPER_MODEL'),
}
model_plan


{'dataset': 'Mozilla Common Voice Yue',
 'train_rows': 500,
 'dev_rows': 100,
 'primary_metric': 'CER',
 'baseline_method': 'Whisper API zero-shot transcription',
 'api_base_url': 'https://api.vectorengine.cn/v1',
 'api_model': 'whisper-1'}

## Run API transcription

Set `VECTORENGINE_API_KEY` in `.env` before running this cell. Begin with `api_rows = 3`, inspect the predictions, then increase to 20 or 100 when the output looks correct.


In [4]:
from openai import OpenAI

api_rows = 20
prediction_path = ROOT / 'outputs' / 'predictions' / f'whisper_api_dev_{api_rows}.csv'

config = WhisperApiConfig.from_env()
client = OpenAI(api_key=config.api_key, base_url=config.base_url)

records = []
for row in dev.head(api_rows).to_dict('records'):
    prediction = transcribe_audio(client, row['audio_path'], config.model)
    records.append(build_prediction_row(row, prediction))

predictions = pd.DataFrame(records)
prediction_path.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(prediction_path, index=False, encoding='utf-8-sig')

summary = {
    'rows': len(predictions),
    'mean_cer': predictions['cer'].mean(),
    'mean_wer': predictions['wer'].mean(),
    'prediction_path': str(prediction_path),
}
summary


{'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'mean_wer': np.float64(0.8833333333333334),
 'prediction_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\predictions\\whisper_api_dev_20.csv'}

In [5]:
display_columns = ['path', 'sentence', 'prediction', 'cer', 'wer']
predictions[display_columns].head(api_rows)


,path,sentence,prediction,cer,wer
0,common_voice_yue_39013334.mp3,學校嘅咩問題，講畀阿媽知得唔得？,"學校嘅咩問題,講俾阿媽知得唔得?",0.066667,0.500000
1,common_voice_yue_39013050.mp3,你鍾唔鍾意男仔？,你鍾唔鍾意男仔?,0.000000,0.000000
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,0.833333,1.000000
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽，係乜嘢驅使你嚟呢度參賽？,"你從來都不參加比賽,是什麼驅使你來這裡參賽?",0.333333,1.000000
4,common_voice_yue_39013110.mp3,輸咗唔好喊，唔好走去搵阿媽求救,"輸了不要哭,不要去找媽媽求救。",0.600000,1.000000
5,common_voice_yue_39013113.mp3,所以我就要飲啤酒,所以我要喝啤酒,0.250000,1.000000
6,common_voice_yue_39013129.mp3,都係手寫算數,還是手寫算數,0.333333,1.000000
7,common_voice_yue_39013130.mp3,叫乜嘢名呢,叫乜嘢名呢?,0.000000,0.000000
8,common_voice_yue_39013034.mp3,呢個真係超搞笑啊！,這個真是超搞笑啊!,0.250000,1.000000
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,0.625000,1.000000


## Use predictions in error analysis

After the API run finishes, open `06_error_analysis.ipynb` and set its prediction CSV path to the file saved above.
